# Calculating Public Services for Wards

Population data from: https://www.elections.in/delhi/mcd-elections/mcd-ward-list-2017.html

In [ ]:
import os
import pickle
from importlib import reload
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, box
import spatial_index_utils
from spatial_index_utils import reproject_gdf, calc_all_services_wards

In [ ]:
reload(spatial_index_utils)

In [ ]:
# WGS 84 / Delhi
epsg_code = 7760

## Import Ward Population

In [ ]:
ward_pop = pd.read_csv('wards_pop.csv')
ward_pop.head()

## Import Wards Shapefile

In [ ]:
ward_shapefile = gpd.read_file('Ward_2017_Fixed.shp')
ward_shapefile.head()

## Merge Ward Shapefile with Ward Population

In [ ]:
ward_shapefile = ward_shapefile.merge(ward_pop, how='inner', left_on='WARD_NO', right_on='Ward No.')
ward_shapefile.head()

In [ ]:
# WARD_NO is the unique ID
len(ward_shapefile['WARD_NO'].unique())

In [ ]:
ward_shapefile.columns

## Check Validity of Ward Shapefile

In [ ]:
# boundary of Delhi
delhi_bounds_filepath = os.path.join('shapefiles', 'delhi_bounds_buffer.shp')

In [ ]:
spatial_index_utils.check_shapefile(gdf=ward_shapefile, gdf_name='wards', 
                                    geom_type='Polygon', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

## Create index column

In [ ]:
ward_shapefile['index'] = ward_shapefile.index

In [ ]:
ward_shapefile.head()

## Import services shapefiles

In [ ]:
# Define filepaths

services_dir = os.path.join('shapefiles', 'Spatial_Index_GIS', 'Public Services')

bank_fp = os.path.join(services_dir, 'Banking', 'Banking.shp')
health_fp = os.path.join(services_dir, 'Health', 'Health.shp')
road_fp = os.path.join(services_dir, 'Major Road', 'Road.shp')
police_fp = os.path.join(services_dir, 'Police', 'Police Station.shp')
ration_fp = os.path.join(services_dir, 'Ration', 'Ration.shp')
school_fp = os.path.join(services_dir, 'School', 'schools7760.shp')
transport_fp = os.path.join(services_dir, 'Transport', 'Transport.shp')

# boundary of Delhi
delhi_bounds_filepath = os.path.join('shapefiles', 'delhi_bounds_buffer.shp')

# Check that all filepaths exist
filepath_list = [bank_fp, health_fp, road_fp, police_fp, ration_fp, school_fp, transport_fp, delhi_bounds_filepath]

for filepath in filepath_list:
    if not os.path.exists(filepath):
        print('{} does not exist'.format(filepath))
        
# Import services
bank = gpd.read_file(bank_fp)
health = gpd.read_file(health_fp)
road = gpd.read_file(road_fp)
police = gpd.read_file(police_fp)
ration = gpd.read_file(ration_fp)
school = gpd.read_file(school_fp)
transport = gpd.read_file(transport_fp)

## Reproject everything to EPSG 7760

In [ ]:
ward_shapefile = reproject_gdf(ward_shapefile, epsg_code)

In [ ]:
bank.crs == health.crs == road.crs == police.crs == ration.crs == school.crs == transport.crs == ward_shapefile.crs

## Test Service Index for Wards

In [ ]:
create_service_index_wards(polygon_gdf=ward_shapefile,
                           point_gdf=health,
                           service_name='health',
                           epsg_code=epsg_code)

## Define Point and Line Services

In [ ]:
# Define all point services as dictionary
# makes it easier to calculate all point
# services with one function
point_services = {'bank': bank,
                  'health': health,
                  'police': police,
                  'ration': ration,
                  'school': school,
                  'transport': transport}

line_services = {'road': road}

In [ ]:
ward_idx = calc_all_services_wards(ward_shapefile, point_services, line_services, epsg_code)

In [ ]:
ward_idx.head()

## Visualizing Results

## Ration Shops

In [ ]:
ward_idx.groupby('ration_count').size()

In [ ]:
ward_idx['ration_count'].plot(kind='hist')

In [ ]:
ward_idx['ration_pcen'].plot(kind='hist')

In [ ]:
ward_idx['ration_idx'].plot(kind='hist')

In [ ]:
ward_idx['school_idx'].plot(kind='hist')

In [ ]:
ward_idx['bank_idx'].plot(kind='hist')

In [ ]:
ward_idx['road_idx'].plot(kind='hist')

## Health

In [ ]:
ward_idx['health_idx'].plot(kind='hist')

In [ ]:
ward_idx['health_pcen'].plot(kind='hist')

In [ ]:
ward_idx['health_count'].plot(kind='hist')

In [ ]:
ward_idx['transport_idx'].plot(kind='hist')

In [ ]:
ward_idx['police_idx'].plot(kind='hist')

In [ ]:
ward_idx['health_count'].unique()

In [ ]:
ward_idx.groupby('health_count').size()

In [ ]:
ward_idx['Total Population'].describe()

## Buffer 
* Pretend there is only one number (not creating multiple variables)
* No neighbors (like ward)

In [ ]:
ward_shapefile['buffer'] = ward_shapefile.buffer(1000)

In [ ]:
ward_shapefile.head()

# Save file

In [ ]:
ward_idx.head()

In [ ]:
ward_idx['geometry']

In [ ]:
ward_idx.drop(columns=['geom_type']).to_file('ward_idx.shp')